# 🎯 Intern Performance Prediction
### Predicting which interns will excel or struggle using Machine Learning
**Model:** Random Forest Regressor  
**Dataset:** 200 Intern Records  
**Target:** Performance Score → Excellent / Average / Needs Improvement

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2 — Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('intern_performance_dataset.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
print(df.info())
print('\n=== Basic Statistics ===')
df.describe()

In [ ]:
# Null value check
print('=== Null Values ===')
print(df.isnull().sum())

# Performance label distribution
print('\n=== Performance Label Distribution ===')
print(df['performance_label'].value_counts())

In [ ]:
# Performance score distribution
plt.figure(figsize=(8, 4))
sns.histplot(df['performance_score'], bins=30, kde=True, color='steelblue')
plt.title('Performance Score Distribution')
plt.xlabel('Performance Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Performance label bar chart
plt.figure(figsize=(6, 4))
df['performance_label'].value_counts().plot(
    kind='bar', color=['green', 'orange', 'red'], edgecolor='black'
)
plt.title('Intern Performance Categories')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 4 — Data Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical columns
le = LabelEncoder()
df['gender']     = le.fit_transform(df['gender'])
df['department'] = le.fit_transform(df['department'])
df['city']       = le.fit_transform(df['city'])

print('Categorical columns encoded successfully!')

# Define features and target
features = [
    'task_completion_rate',
    'avg_completion_time',
    'feedback_rating',
    'attendance_rate',
    'tasks_assigned',
    'tasks_completed',
    'communication_score',
    'initiative_score',
    'late_submissions',
    'trainings_attended',
    'mentor_rating',
    'gender',
    'department',
    'city'
]

x = df[features]
y = df['performance_score']

print('Features shape:', x.shape)
print('Target shape  :', y.shape)

In [ ]:
# Train test split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Feature scaling
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)

print('Train size:', x_train_scaled.shape)
print('Test size :', x_test_scaled.shape)

## Step 5 — Train Random Forest Model

In [ ]:
# Train model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

rf_model.fit(x_train_scaled, y_train)
print('Model trained successfully!')

## Step 6 — Model Evaluation

In [ ]:
# Predictions
y_pred_train = rf_model.predict(x_train_scaled)
y_pred_test  = rf_model.predict(x_test_scaled)

# Scores
print('=' * 45)
print('        MODEL PERFORMANCE RESULTS')
print('=' * 45)
print(f'Train R2 Score : {r2_score(y_train, y_pred_train):.4f}')
print(f'Test  R2 Score : {r2_score(y_test,  y_pred_test):.4f}')
print(f'MAE            : {mean_absolute_error(y_test, y_pred_test):.4f}')
print(f'RMSE           : {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}')
print('=' * 45)

# Cross validation
cv_scores = cross_val_score(rf_model, x_train_scaled, y_train, cv=5, scoring='r2')
print(f'Cross-Val R2   : {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_test, alpha=0.6, color='steelblue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Score')
plt.ylabel('Predicted Score')
plt.title('Actual vs Predicted Performance Score')
plt.legend()
plt.tight_layout()
plt.show()

## Step 7 — Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature'   : features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance - Which factors affect performance most?')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## Step 8 — Predict & Classify All Interns

In [ ]:
# Predict on full dataset
x_all_scaled = scaler.transform(df[features])
df['predicted_score'] = rf_model.predict(x_all_scaled).round(2)

# Classify interns
def classify_intern(score):
    if score >= 75:
        return 'Excellent'
    elif score >= 55:
        return 'Average'
    else:
        return 'Needs Improvement'

df['predicted_label'] = df['predicted_score'].apply(classify_intern)

print('=== Category Distribution ===')
print(df['predicted_label'].value_counts())

print('\n=== Sample Predictions ===')
df[['intern_id', 'predicted_score', 'predicted_label']].head(15)

In [ ]:
# Top 5 interns
print('=== TOP 5 INTERNS ===')
print(df.nlargest(5, 'predicted_score')[['intern_id', 'predicted_score', 'predicted_label']].to_string(index=False))

print('\n=== BOTTOM 5 INTERNS ===')
print(df.nsmallest(5, 'predicted_score')[['intern_id', 'predicted_score', 'predicted_label']].to_string(index=False))

In [ ]:
# Final category chart
plt.figure(figsize=(6, 4))
counts = df['predicted_label'].value_counts()
colors = ['green' if l == 'Excellent' else 'orange' if l == 'Average' else 'red' 
          for l in counts.index]
counts.plot(kind='bar', color=colors, edgecolor='black')
plt.title('Final Intern Performance Prediction')
plt.xlabel('Category')
plt.ylabel('Number of Interns')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('\n Project Complete!')